In [ ]:
%pip install spectral
%pip install rasterio

# Rasterização

In [10]:
from osgeo import gdal
from osgeo import ogr
import geopandas as gpd
import rasterio as rio
import numpy as np

In [ ]:
raster = gdal.Open('/content/drive/MyDrive/PDIPy/vacaria/vacaria_recorte_allbands.tif')
vetor = ogr.Open('/content/drive/MyDrive/PDIPy/vacaria/pontos_agua.shp')

#1 - telas
#2 - cultivos
#3 - mata
#4 - solo
#5 - agua
#6 - imoveis

In [4]:
layer = vetor.GetLayer()

In [5]:
transform = raster.GetGeoTransform()

In [6]:
driver = gdal.GetDriverByName('GTiff')
raster_pts = driver.Create('raster_pts.tif', raster .RasterXSize, raster .RasterYSize, 1, gdal.GDT_Int16)
raster_pts.SetGeoTransform(transform)

0

In [7]:
gdal.RasterizeLayer(raster_pts, [1], layer, options=['ATTRIBUTE=class'])
raster_pts.GetRasterBand(1).SetNoDataValue(0.0)
raster_pts = None





# KNN

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio as rio
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

In [17]:
src = rio.open('/content/drive/MyDrive/PDIPy/vacaria/vacaria_recorte_allbands.tif')
b2 = src.read(1)
b3 = src.read(2)
b4 = src.read(3)
b5 = src.read(4)
b6 = src.read(5)
b7 = src.read(6)
b8 = src.read(7)
b8A = src.read(8)
b11 = src.read(9)
b12 = src.read(10)

img = np.dstack([b2, b3, b4, b5, b12])

In [18]:
meta = src.profile

In [19]:
with rio.open('raster_pts.tif') as src2:
    raster_pts = src2.read(1)

In [ ]:
np.unique(raster_pts)

In [21]:
X = img[raster_pts > 0]
y  = raster_pts[raster_pts > 0]

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y.ravel(), test_size = 0.3, stratify = y.ravel())

In [ ]:
knn = KNeighborsClassifier(n_neighbors=10, n_jobs=-1)

knn.fit(X_train, y_train)

In [28]:
img2d = img.reshape(img.shape[0]*img.shape[1], img.shape[2])

pred_knn = knn.predict(img2d)

pred_knn_final = pred_knn.reshape(img[:,:,0].shape)

In [ ]:
plt.figure(figsize=(12,8))
plt.imshow(pred_knn_final)
plt.show()

In [ ]:
print(np.unique(pred_knn))

[1 2 3 4 5 6]


In [ ]:
meta = src.profile

with rio.open('knn.tif', 'w', **meta) as src:
  src.write(pred_knn_final,1)

# RF

In [8]:
from sklearn.ensemble import RandomForestClassifier

In [11]:
with rio.open('raster_pts.tif') as src2:
    raster_pts = src2.read(1)

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y.ravel(), test_size = 0.3, stratify = y.ravel())

In [ ]:
rf = RandomForestClassifier(n_estimators=500, max_depth=1000, n_jobs=-1, oob_score=True)

rf.fit(X_train, y_train)

In [30]:
pred_rf = rf.predict(img2d)

pred_rf_final = pred_rf.reshape(img[:,:,0].shape)

In [33]:
with rio.open(
    'classified_rf.tif',
    'w',
    driver='GTiff',
    height=pred_rf_final.shape[0],
    width=pred_rf_final.shape[1],
    count=1,
    dtype=pred_rf_final.dtype,
    crs=src.crs,
    transform=src.transform
) as dst:
    dst.write(pred_rf_final, 1)

In [ ]:
plt.figure(figsize=(12,8))
plt.imshow(pred_rf_final)

In [ ]:
with rio.open('rf.tif', 'w', **meta) as src:
  src.write(pred_rf_final,1)

# MLP

In [ ]:
from spectral import *
import matplotlib.pyplot as plt
import rasterio as rio
import numpy as np

In [ ]:
with rio.open('raster_pts.tif') as src2:
    gt = src2.read(1)

In [ ]:
np.unique(gt)

array([0, 1, 2, 3, 4, 5, 6], dtype=int16)

In [ ]:
classes_mlp = create_training_classes(img, gt)

inputs = img.shape[2]
outputs = len(classes_mlp)

In [ ]:
mlp = PerceptronClassifier([inputs, 20, 10, outputs])

mlp.train(classes_mlp, 20, clip=0., accuracy=100)

classified_mlp = mlp.classify(img)

In [ ]:
plt.figure(figsize=(12,8))
plt.imshow(classified_mlp)

In [ ]:
meta = src.profile

with rio.open('mlp.tif', 'w', **meta) as src:
  src.write(c,1)



# Acuracias

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, cohen_kappa_score

In [ ]:
# KNN

pred_knn = knn.predict(X_test)
print(classification_report(y_test, pred_knn))
print('Matriz de confusao: ', confusion_matrix(y_test, pred_knn))
print('Acurácia: ', accuracy_score(y_test, pred_knn))
print('Kappa: ', cohen_kappa_score(y_test, pred_knn))

In [ ]:
# RF

pred_rf = rf.predict(X_test)
print(classification_report(y_test, pred_rf))
print('Matriz de confusao: \n', confusion_matrix(y_test, pred_rf))
print(rf.oob_score_*100) #acuracia
print('Acurácia: ', accuracy_score(y_test, pred_rf))
print('Kappa: ', cohen_kappa_score(y_test, pred_rf))

In [ ]:
# MLP

pred_mlp = mlp.predict(X_test)
print(classification_report(y_test, pred_mlp))
print('Matriz de confusao: ', confusion_matrix(y_test, pred_mlp))
print('Acurácia: ', accuracy_score(y_test, pred_mlp))
print('Kappa: ', cohen_kappa_score(y_test, pred_mlp))

In [ ]:
import numpy as np
from spectral import PerceptronClassifier, create_training_classes
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, cohen_kappa_score

# gt: raster de rótulos (1..N, 0 = sem rótulo)
# img: array (rows, cols, bands)

# 1) Pegar coordenadas e labels dos pixels rotulados
coords = np.column_stack(np.where(gt > 0))   # array de shape (n_samples, 2) com (row, col)
labels = gt[gt > 0]

# 2) dividir em treino/teste — OBS: fazemos split sobre as coords (posições)
coords_train, coords_test, y_train, y_test = train_test_split(
    coords, labels, test_size=0.3, stratify=labels, random_state=42
)

# 3) criar um raster de treino que contenha apenas os rótulos de treino
gt_train = np.zeros_like(gt)
for (r, c), lab in zip(coords_train, y_train):
    gt_train[r, c] = int(lab)

# 4) criar classes de treino para o spectral e treinar o MLP
classes_mlp = create_training_classes(img, gt_train)

inputs = img.shape[2]
outputs = len(np.unique(y_train))  # ou outra forma de obter número de classes esperadas
# ajustar a arquitetura conforme quiser
mlp = PerceptronClassifier([inputs, 20, 10, outputs])

# treinar (ajuste os parâmetros de train conforme necessário)
mlp.train(classes_mlp, clip=0., accuracy=100)  # ou apenas mlp.train(classes_mlp, 20, ...)

# 5) classificar a imagem inteira
classified = mlp.classify(img)   # retorna array (rows, cols) com rótulos preditos

# 6) extrair previsões nos pixels de teste usando coords_test
preds_test = np.array([ classified[r, c] for r, c in coords_test ])

# 7) calcular métricas
print(classification_report(y_test, preds_test))
print("Matriz de Confusão:\n", confusion_matrix(y_test, preds_test))
print("Acurácia:", accuracy_score(y_test, preds_test))
print("Kappa:", cohen_kappa_score(y_test, preds_test))


In [ ]:
plt.figure(figsize=(12,8))
plt.imshow(classified)